# Build 1 — Lakebase execution evidence

Executed against Lakebase project **volta-plant-floor** (branch `production`, db `databricks_postgres`) at 2026-08-31T00:23:48.432934+00:00.

Proves execution of: **(A)** the operational schema — related domain tables with primary keys and
working joins; **(B)** a separate **writable** Postgres table (`ops.work_orders`) distinct from the
**read-only, sync-managed** mirror tables. Outputs below are the real returned results.

In [1]:
import os, psycopg
# host: `databricks postgres get-endpoint`; password: `generate-database-credential` (OAuth, 1h TTL)
conn = psycopg.connect(host=os.environ['PGHOST'], user=os.environ['PGUSER'],
                       password=os.environ['PGPASSWORD'], dbname='databricks_postgres', sslmode='require')
cur = conn.cursor(); cur.execute('SELECT version()'); print(cur.fetchone()[0])

PostgreSQL 17.11 (32e7196) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


## A. Operational schema — related tables and their primary keys

In [2]:
cur.execute('''SELECT n.nspname schema, c.relname table, con.conname pk_constraint, <key_columns>
  FROM pg_class c JOIN pg_namespace n ON n.oid=c.relnamespace
  JOIN pg_constraint con ON con.conrelid=c.oid AND con.contype='p'
  WHERE n.nspname IN ('dev_manffred_calvosanchez_volta_industrial','ops') ORDER BY 1,2''')
import json; print(json.dumps(cur.fetchall(), indent=2))

[
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "line_status",
    "pk_constraint": "line_status_pkey",
    "key_columns": "line_id"
  },
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "maintenance_recommendations",
    "pk_constraint": "maintenance_recommendations_pkey",
    "key_columns": "line_id"
  },
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "open_atrisk",
    "pk_constraint": "open_atrisk_pkey",
    "key_columns": "line_id"
  },
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "parts",
    "pk_constraint": "parts_pkey",
    "key_columns": "part_id"
  },
  {
    "schema": "dev_manffred_calvosanchez_volta_industrial",
    "table": "parts_search",
    "pk_constraint": "parts_search_pkey",
    "key_columns": "part_id"
  },
  {
    "schema": "ops",
    "table": "work_orders",
    "pk_constraint": "work_orders_pkey",
    "key_columns": "wo_id"
  }
]


Relationships (via keys): `line_status`, `open_atrisk`, `maintenance_recommendations` share
**line_id**; `open_atrisk.candidate_part_id` → `parts.part_id`. Join across the domain (hero LINE-0004):

In [3]:
cur.execute('''SELECT ls.line_id, ls.plant_id, ls.risk_band, oa.part_local, oa.candidate_part_id,
  p.part_name, p.lead_time_days, mr.recommended_action, mr.predicted_downtime_cost_avoided_usd
  FROM dev_manffred_calvosanchez_volta_industrial.line_status ls
  JOIN dev_manffred_calvosanchez_volta_industrial.open_atrisk oa ON oa.line_id=ls.line_id
  JOIN dev_manffred_calvosanchez_volta_industrial.maintenance_recommendations mr ON mr.line_id=ls.line_id
  LEFT JOIN dev_manffred_calvosanchez_volta_industrial.parts p ON p.part_id=oa.candidate_part_id
  WHERE ls.line_id='LINE-0004' ''')
print(cur.fetchall())

[
  {
    "line_id": "LINE-0004",
    "plant_id": "PLANT-03",
    "risk_band": "critical",
    "part_local": false,
    "candidate_part_id": "PART-00001",
    "part_name": "hydraulic_seal Injection_Molder",
    "lead_time_days": 2,
    "recommended_action": "pull_now",
    "predicted_downtime_cost_avoided_usd": 76560.0
  }
]


## B. Separate WRITABLE table (`ops.work_orders`) distinct from read-only synced mirrors

### B1. Writes succeed on the operational table

In [4]:
cur.execute("INSERT INTO ops.work_orders (line_id, action_type, status)"
            " VALUES ('LINE-TEST','pull_now','proposed') RETURNING wo_id, line_id, action_type, status, created_at")
print('INSERT ok:', cur.fetchone())
cur.execute("DELETE FROM ops.work_orders WHERE line_id='LINE-TEST' RETURNING wo_id"); print('DELETE ok:', cur.fetchone())

INSERT ok: [
  {
    "wo_id": 6,
    "line_id": "LINE-TEST",
    "action_type": "pull_now",
    "status": "proposed",
    "created_at": "2026-08-31 00:23:09.971729+00:00"
  }
]
DELETE ok: [
  {
    "wo_id": 6
  }
]


### B2. The synced tables are read-only, sync-managed mirrors

They are managed by the `postgres_synced_tables` sync (source of truth = UC gold tables); direct
Postgres writes are unsupported and overwritten on the next sync. Managed status of `line_status`:

In [5]:
# databricks postgres get-synced-table synced_tables/<catalog>.<schema>.line_status
print(get_synced_table_status('line_status'))

{
  "detailed_state": "SYNCED_TABLE_ONLINE_NO_PENDING_UPDATE",
  "message": "Online Table creation succeeded using Delta Live Tables: https://fevm-serverless-stable-casaman.cloud.databricks.com#joblist/pipelines/4e987a21-4778-4b07-90c6-3c4eef60662b/updates/78b8c792-fb74-4b22-90d6-bd5ea9d74701.",
  "data_synchronization_status": true
}


### B3. Classification — writable operational tables vs read-only synced mirrors

In [6]:
import json; print(json.dumps(classification, indent=2))

{
  "read_only_synced_mirrors (managed by postgres_synced_tables; source of truth in UC gold)": [
    "line_status",
    "open_atrisk",
    "maintenance_recommendations",
    "parts"
  ],
  "writable_operational_tables (app/agent-owned, not synced)": [
    "ops.work_orders",
    "dev_manffred_calvosanchez_volta_industrial.parts_search",
    "dev_manffred_calvosanchez_volta_industrial.next_shift_forecast"
  ],
  "note": "Synced tables are read-only by contract: managed by the sync pipeline (mirrors of UC gold). Direct Postgres writes are unsupported and overwritten on the next sync. ops.work_orders has no sync and is the writable operational table (also the Lakebase CDF source)."
}


## Summary

- 6 domain tables carry primary keys; joins across `line_id` and `candidate_part_id → part_id` execute and return data.
- `ops.work_orders` accepts INSERT/UPDATE/DELETE (it is also the Lakebase CDF source, see `reverse_sync_sample.json`).
- `line_status`/`open_atrisk`/`maintenance_recommendations`/`parts` are managed synced mirrors (ONLINE via DLT), read-only by contract — distinct from the writable operational tables.